In [1]:
def xmatched_lists(cases):
    """Cross match SPOC and QLP lists using corresponding csv files"""
    import pandas as pd

    #dataframes for SPOC 2-min and QLP targets
    if cases=='targets':
        spoc_targets = pd.read_csv("targetLists/all_targets_S056_v1.csv",
                                   skiprows=5)
    elif cases=='tces':
        spoc_tces = pd.read_csv(
            "targetLists/tess2022245180045-s0056-s0056_dvr-tcestats.csv",
            skiprows=6)
    elif cases=='FTL':
        spoc_FTL = pd.read_csv("targetLists/FTL_s0056.csv")
        
    qlp_targets = pd.read_csv("targetLists/qlp_s0056.csv")
    
    #change name of the TIC ID column to match that of the spoc dataframe
    qlp_targets.rename(columns={'#TIC_ID': 'TICID'}, inplace=True)
    #Similarly for SPOC tces if used
    if cases=='tces':
        spoc_tces.rename(columns={'ticid': 'TICID'}, inplace=True)
    if cases=='FTL':
        spoc_FTL.rename(columns={'#TIC_ID': 'TICID'}, inplace=True)
    
    #cross matched both dataframes based on the ticid column
    if cases=='targets':
        xmatched_ticids=pd.merge(spoc_targets, qlp_targets, on='TICID', 
                                 how='inner')
    elif cases=='tces':
        xmatched_ticids=pd.merge(spoc_tces, qlp_targets, on='TICID', 
                                 how='inner')
    elif cases=='FTL':
        spoc_FTL
        xmatched_ticids=pd.merge(spoc_FTL, qlp_targets, on='TICID', 
                                 how='inner')
    else:
        print("For cross-matching, please enter 'targets', 'tces', or 'FTL'.")
        sys.exit(1)
    
    return xmatched_ticids

In [2]:
def find_dir(directory, pattern):
    """Finds directories matching a pattern within a given directory."""
    import os, glob
    
    matching_dirs = []

    for root, dirs, files in os.walk(directory):
        for dir_name in dirs:
            if glob.fnmatch.fnmatch(dir_name, pattern):
                matching_dirs.append(os.path.join(root, dir_name))

    return matching_dirs
    
def load_QLP_SPOC_lcs():
    """Load SPOC and QLP light curves"""
    import numpy.ma as ma
    import warnings
    
    cache_dir=lk.config.get_cache_dir()+'/mastDownload/'
    hlsp_qlp_folder='HLSP/QLP'
    hlsp_ftl_folder='HLSP/FTL'
    spoc_twomin_folder='TESS'
    
    qlp_dir = cache_dir+hlsp_qlp_folder
    qlp_ticid_dir = find_dir(qlp_dir, f'*{ID}*')
    if not qlp_ticid_dir:
        #Obtain QLP normalized light curve using the TIC ID with the search tool
        search_results_qlp=lk.search_lightcurve(
            target=TICID, exptime=exptime_qlp, mission='TESS', author='QLP', 
            sector=sector)
        for entry in search_results_qlp:
            id_entry=str(entry.target_name[0])
            if id_entry==ID:
                try:
                    qlp_lc_normalized=entry.download(
                        quality_bitmask=quality_bitmask, 
                        flux_column='sys_rm_flux',
                        flux_err_column='det_flux_err')
                except:
                    print(f'Download to {qlp_dir} failed.')
                obs_id=search_results_qlp.table['obs_id']    
                for obs in obs_id:
                    if ID in obs:
                        shutil.move(cache_dir+f'HLSP/{obs}',qlp_dir)
    
    qlp_ticid_dir = find_dir(qlp_dir, f'*{ID}*')
    qlp_fileroot = qlp_ticid_dir[0].split('/')[-1]
    qlp_filename = qlp_ticid_dir[0]+'/'+qlp_fileroot+'.fits'
    qlp_lc_normalized=lk.read(qlp_filename, quality_bitmask=quality_bitmask)

    if cases=='targets' or cases=='tces':
        spoc_dir = cache_dir+spoc_twomin_folder
        spoc_ticid_dir = find_dir(spoc_dir, f'*{ID}*')
        if not spoc_ticid_dir:
            #Obtain the SPOC counterpart in 2-min cadence
            search_results_spoc=lk.search_lightcurve(
                target=TICID, exptime=exptime_spoc, mission='TESS', 
                author='SPOC', sector=sector)
            for entry in search_results_spoc:
                id_entry=str(entry.target_name[0])
                if id_entry==ID:
                    try:
                        spoc_lc_normalized=entry.download(
                            quality_bitmask=quality_bitmask,
                            flux_column='pdcsap_flux')
                    except:
                        print(f'Download to {spoc_dir} failed.')
                        
    elif cases=='FTL':
        spoc_dir = cache_dir+hlsp_ftl_folder
        spoc_ticid_dir = find_dir(spoc_dir, f'*{ID}*')
        if not spoc_ticid_dir:
            #Obtain the SPOC counterpart in 2-min cadence
            search_results_ftl=lk.search_lightcurve(
                target=TICID, exptime=exptime_ftl, mission='TESS', 
                author='TESS-SPOC', sector=sector)
            for entry in search_results_ftl:
                id_entry=str(entry.target_name[0])
                if id_entry==ID:
                    try:
                        spoc_lc_normalized=entry.download(
                            quality_bitmask=quality_bitmask,
                            flux_column='pdcsap_flux')
                    except:
                        print(f'Download to {spoc_dir} failed.')
                    obs_id=search_results_ftl.table['obs_id']
                    for obs in obs_id:
                        if ID in obs:
                            shutil.move(cache_dir+f'HLSP/{obs}',spoc_dir)
                    
    spoc_ticid_dir = find_dir(spoc_dir, f'*{ID}*')
    spoc_fileroot = spoc_ticid_dir[0].split('/')[-1]
    spoc_filename = spoc_ticid_dir[0]+'/'+spoc_fileroot+'_lc.fits'
    if cases=='FTL':
        spoc_filename = spoc_filename.replace('tp_lc.fits','lc.fits')
    spoc_lc=lk.read(spoc_filename, quality_bitmask=quality_bitmask)
    
    #Other QLP light curves for testing purposes
    #Note: there is no sap_flux_err column; but the default flux appears to be
    #sap_flux with the errors of det_flux_err; so this is not changed here
    qlp_lc_sap=lk.read(qlp_filename, flux_column='sap_flux',
                       quality_bitmask=quality_bitmask)
    qlp_lc_bkg=lk.read(qlp_filename, flux_column='sap_bkg',
                       flux_err_column='sap_bkg_err',
                       quality_bitmask=quality_bitmask)

    #Other SPOC light curves for testing purposes
    spoc_lc_normalized=spoc_lc.normalize('unscaled')

    #Note: for spoc, the err column is added automatically; do not add
    spoc_lc_sap=lk.read(spoc_filename, flux_column='sap_flux',
                        quality_bitmask=quality_bitmask)
    spoc_lc_bkg=lk.read(spoc_filename, flux_column='sap_bkg',
                        quality_bitmask=quality_bitmask)
        
    #Calculating the difference between the SPOC and QLP backgroung fluxes
    spoc_median_bkg=np.float32(stats.median(spoc_lc_bkg['flux'].value))
    qlp_median_bkg=stats.median(qlp_lc_bkg['flux'])
    median_bkg_diff=spoc_median_bkg-qlp_median_bkg
    print(f'The median flux background difference between SPOC', 
          f'and QLP is {median_bkg_diff:.2f} e/s\n', file=file)

    #Obtain the TESS magnitude from the lc object
    Tmag=qlp_lc_normalized.meta['TESSMAG']

    #Convert TESS magnitude (Tmag) into flux. See: 
    #https://tess.mit.edu/public/tesstransients/pages/
    #readme.html#flux-calibration
    #Tmag=-2.5log10(cts_per_s) + zeropoint; where cts_per_s is counts per second
    #With zeropoint and its uncertainty in mag
    zp=20.44
    zp_uncertainty=0.05
    
    flux=pow(10,(zp-Tmag)/2.5)
    flux_uncertainty=flux*(2/5)*np.log(10)*zp_uncertainty
    flux_up=flux+flux_uncertainty
    flux_down=flux-flux_uncertainty
    flux_2up=flux+2*flux_uncertainty
    flux_2down=flux-2*flux_uncertainty
    #flux_up=pow(10,(zp+zp_uncertainty-Tmag)/2.5)
    #flux_down=pow(10,(zp-zp_uncertainty-Tmag)/2.5)
    #flux_2up=pow(10,(zp+2*zp_uncertainty-Tmag)/2.5)
    #flux_2down=pow(10,(zp-2*zp_uncertainty-Tmag)/2.5)
    crowdsap=spoc_lc.meta['CROWDSAP']
    
    print(f'The flux derived from a TESS magnitude', 
          f'of {Tmag:.2f} is {flux:.2f} +',
          f'{flux_up-flux:.2f} - {flux-flux_down:.2f} e/s',
          file=file)
    print(f'The crowding metric is {crowdsap:.2f}', file=file)

    #To ignore the userwarning generated right below if there are nans
    warnings.filterwarnings('ignore', category=UserWarning, append=True)
    #To avoid error in testing below due to nans (which appear in rare cases 
    #such as 1868, 2493 in 'targets')
    qlp_lc_normalized = qlp_lc_normalized[~np.isnan(qlp_lc_normalized.flux)]
    
    #Unnormalize the 'normalized flux' of the qlp light curve 
    #with the flux of Tmag
    flux_unit=spoc_lc.flux.unit
    qlp_flux_unnormalized=qlp_lc_normalized.sys_rm_flux*flux*flux_unit
    qlp_det_flux_err_unnormalized=qlp_lc_normalized.flux_err*flux*flux_unit
    
    qlp_lc=copy.deepcopy(qlp_lc_normalized)
    qlp_lc.flux=qlp_flux_unnormalized
    qlp_lc.flux_err=qlp_det_flux_err_unnormalized
    qlp_lc.meta['NORMALIZED']=False

    #Testing the unnormalization by reversing it
    median=stats.median(qlp_flux_unnormalized)
    normalized_flux=np.float32([flux/median for flux in qlp_flux_unnormalized])
    #Residuals in ppm of the qlp-normalized flux and 
    #our normalized 'unnormalized normalized flux'
    residual=np.float32(qlp_lc['sys_rm_flux']-normalized_flux)*1e6
    print('Residuals in ppm of our unnormalization test (mean and stdev)',
          f"{stats.mean(residual):.2f} \u00B1 {stats.stdev(residual):.2f}\n",
          file=file)
    
    return qlp_lc_normalized, qlp_lc, qlp_lc_sap, qlp_lc_bkg,\
    spoc_lc_normalized, spoc_lc, spoc_lc_sap, spoc_lc_bkg, median_bkg_diff,\
    flux_up, flux_down, flux_2up, flux_2down

In [3]:
def compare_and_mask(table_large, table_small, compare_column):
    """
    Compares a specific column between two Astropy tables of potentially 
    different shapes and returns a mask to remove rows from the larger table 
    where the column value differs from the smaller table. 
    
    Args:
        table_large (astropy.table.Table): The larger table.
        table_small (astropy.table.Table): The smaller table to compare against.
        compare_column (str): Name of the column to compare. 
    
    Returns:
        np.ndarray: A boolean mask where True indicates rows in table_large 
        that should be kept. 
    """
    
    # Extract comparison column from both tables
    large_col = table_large[compare_column]
    small_col = table_small[compare_column]
    
    # Use broadcasting to compare values
    mask = (large_col[:, None] == small_col)
    
    # If comparing to a single value, check if all values in the smaller table 
    # match
    if len(small_col) == 1:
        mask = np.all(mask, axis=1) 
    else:
        # Ensure the mask is based on the larger table's length
        mask = np.any(mask, axis=1) 
    
    return mask

def calculate_stats():
    """Un-normalize the QLP light curve and calculate statistics and metrics 
    from them such as CDPPs"""
    import pandas as pd
    from astropy.utils.masked import Masked

    global qlp_lc, qlp_lc_normalized, spoc_lc
    
    savgol_window_qlp=101
    if cases=='targets' or cases=='tces':
        transit_duration_spoc=int(3600/exptime_spoc)
    elif cases=='FTL':
        transit_duration_spoc=int(3600/exptime_ftl)
    transit_duration_qlp=int(3600/exptime_qlp)
    w_ratio=transit_duration_spoc/transit_duration_qlp
    savgol_window_spoc=round(w_ratio*savgol_window_qlp)

    if cases=='FTL':
        #For comparison purposes: selecting the same cadences for QLP as FTL
        mask=compare_and_mask(qlp_lc,spoc_lc,'cadenceno')
        qlp_lc=qlp_lc[mask]
        qlp_lc_normalized=qlp_lc_normalized[mask]
        mask=compare_and_mask(spoc_lc,qlp_lc,'cadenceno')
        spoc_lc=spoc_lc[mask]
        #and also applying the mask of FTL to QLP
        qlp_lc.flux=Masked(qlp_lc.flux)
        qlp_lc['flux'].mask = spoc_lc['flux'].mask
        qlp_lc_normalized.flux=Masked(qlp_lc_normalized.flux)
        qlp_lc_normalized['flux'].mask = spoc_lc['flux'].mask
        
    #SPOC 1h sgCDPP and CDPP
    spoc_sgcdpp_1h=spoc_lc.estimate_cdpp(transit_duration=transit_duration_spoc,
                                         savgol_window=savgol_window_spoc)
    spoc_sgcdpp_1h_normalized=spoc_lc_normalized.estimate_cdpp(
        transit_duration=transit_duration_spoc,
        savgol_window=savgol_window_spoc)
    spoc_sgcdpp_0_5h=spoc_lc.estimate_cdpp(
        transit_duration=int(transit_duration_spoc/2),
        savgol_window=savgol_window_spoc)
    spoc_sgcdpp_2h=spoc_lc.estimate_cdpp(
        transit_duration=2*transit_duration_spoc,
        savgol_window=savgol_window_spoc)
    spoc_cdpp_0_5h=spoc_lc.meta['CDPP0_5']
    spoc_cdpp_0_5h=float(pd.to_numeric(str(spoc_cdpp_0_5h), errors='coerce'))
    spoc_cdpp_1h=spoc_lc.meta['CDPP1_0']
    spoc_cdpp_1h=float(pd.to_numeric(str(spoc_cdpp_1h), errors='coerce'))
    
    #QLP 1h sgCDPP
    qlp_sgcdpp_1h=qlp_lc.estimate_cdpp(transit_duration=transit_duration_qlp,
                                    savgol_window=savgol_window_qlp)
    qlp_sgcdpp_1h_normalized=qlp_lc_normalized.estimate_cdpp(
        transit_duration=transit_duration_qlp,
        savgol_window=savgol_window_qlp)
    qlp_sgcdpp_0_5h=qlp_lc.estimate_cdpp(
        transit_duration=int(transit_duration_qlp/2),
        savgol_window=savgol_window_qlp)
    qlp_sgcdpp_2h=qlp_lc.estimate_cdpp(
        transit_duration=2*transit_duration_qlp,
        savgol_window=savgol_window_qlp)
    
    print(f'The SPOC 1h sgCDPP is {spoc_sgcdpp_1h:.2f}',
          f'   (normalized check: {spoc_sgcdpp_1h_normalized:.2f})', file=file)
    print(f'The QLP  1h sgCDPP is {qlp_sgcdpp_1h:.2f}',
          f'   (normalized check: {qlp_sgcdpp_1h_normalized:.2f})\n', file=file)

    print(f'The SPOC 0.5h sgCDPP is {spoc_sgcdpp_0_5h:.2f}', file=file)
    print(f'The SPOC 2h sgCDPP is   {spoc_sgcdpp_2h:.2f}\n', file=file)

    print(f'The QLP  0.5h sgCDPP is {qlp_sgcdpp_0_5h:.2f}', file=file)
    print(f'The QLP  2h sgCDPP is   {qlp_sgcdpp_2h:.2f}\n', file=file)
    
    print(f'The SPOC 1h CDPP is   {spoc_cdpp_1h:.2f} ppm',
          f'   (0.5h CDPP: {spoc_cdpp_0_5h:.2f} ppm)', file=file)

    return

In [4]:
def plot_QLP_SPOC_tests():
    """Plot QLP light curve and a set of comparisons with SPOC light curve"""
    import matplotlib.pyplot as plt
    
    #QLP unnormalized flux
    qlp_ax=qlp_lc.scatter(ylabel='Flux [$\\mathrm{e^{-}\\,s^{-1}}$]')
    filename=f"{TICID}"+'_QLP_flux_unnormalized.pdf'
    plt.savefig(directory+filename)
    plt.close()
    
    #Comparing SPOC and QLP unnormalized fluxes
    qlp_ax=qlp_lc.scatter(label='QLP systematics-removed flux')
    spoc_lc.scatter(ax=qlp_ax,label='SPOC PDCSAP flux')
    plt.axhline(y=flux_up, color='r', linestyle='--')
    plt.axhline(y=flux_down, color='r', linestyle='--')
    plt.axhline(y=flux_2up, color='g', linestyle='--')
    plt.axhline(y=flux_2down, color='g', linestyle='--')
    filename=f"{TICID}"+'_SPOC_QLP_fluxes_unnormalized.pdf'
    plt.savefig(directory+filename)
    plt.close()
    
    #Comparing SPOC and QLP normalized fluxes
    qlp_ax_normalized=qlp_lc_normalized.scatter(
        label='QLP systematics-removed flux')
    spoc_lc_normalized.scatter(ax=qlp_ax_normalized,label='SPOC PDCSAP flux')
    filename=f"{TICID}"+'_SPOC_QLP_fluxes_normalized.pdf'
    plt.savefig(directory+filename)
    plt.close()

    #Comparing SPOC and QLP SAP background fluxes
    qlp_ax_bkg=qlp_lc_bkg.scatter(label='QLP SAP BKG flux')
    spoc_lc_bkg.scatter(ax=qlp_ax_bkg,label='SPOC SAP BKG flux')
    filename=f"{TICID}"+'_SPOC_QLP_SAP_bkg_fluxes_unnormalized.pdf'
    plt.savefig(directory+filename)
    plt.close()
    
    #Comparing SPOC and QLP normalized SAP fluxes
    spoc_lc_sap_normalized=spoc_lc_sap.normalize('unscaled')
    qlp_ax_sap=qlp_lc_sap.scatter(label='QLP SAP flux')
    spoc_lc_sap_normalized.scatter(ax=qlp_ax_sap,label='SPOC SAP flux')
    filename=f"{TICID}"+'_SPOC_QLP_SAP_fluxes_normalized.pdf'
    plt.savefig(directory+filename)
    plt.close()
    
    #Comparing SPOC and QLP total fluxes
    spoc_lc_total=copy.deepcopy(spoc_lc)
    qlp_lc_total=copy.deepcopy(qlp_lc)
    spoc_lc_total['flux']=spoc_lc_total['flux']+spoc_lc_bkg['flux']
    qlp_lc_total['flux']=qlp_lc_total['flux']+qlp_lc_bkg['flux']
    qlp_lc_total_ax=qlp_lc_total.scatter(label='QLP total flux')
    spoc_lc_total.scatter(ax=qlp_lc_total_ax,label='SPOC total flux')
    filename=f"{TICID}"+'_SPOC_QLP_total_fluxes_unnormalized.pdf'
    plt.savefig(directory+filename)
    plt.close()
    
    #Comparing SPOC and QLP+median_background_difference unnormalized fluxes
    qlp_ax_plus_median_bkg_diff=(qlp_lc+median_bkg_diff).scatter(
        label='QLP systematics-removed flux + median BKG difference')
    spoc_lc.scatter(ax=qlp_ax_plus_median_bkg_diff,label='SPOC PDCSAP flux')
    filename=f"{TICID}"+'_SPOC_QLP_plus_median_bkg_diff_fluxes_unnormalized.pdf'
    plt.savefig(directory+filename)
    plt.close()

    return

In [5]:
import lightkurve as lk
import statistics as stats
import os, copy, shutil, sys
import numpy as np
from pathlib import Path
from IPython.display import IFrame, display, clear_output

quality_bitmask='default'
exptime_qlp=200
exptime_spoc=120
exptime_ftl=200
sector=56

#These ticids did not have a SPOC light curve produced according to the DRN of 
#sector 56 due being bright stars (Tmag <~1.8) with large pixel stamps that were 
#not processed in the photometric pipeline.
if sector==56:
    excluded_ticids=['436774002', '260614141', '229540730', '195554360']
else:
    print(f'Use the DRN of sector {sector} to add an exclude list')
    sys.exit(0)
    
#Choose between 'targets' (2-min), 'tces' (2-min), 'FTL' (200-sec)
cases='FTL'
#This is to utilize below output from 2-min cases that have already been run 
twomin_options=['targets','tces']
#xmatched_ticids=xmatched_lists(targets)
xmatched_ticids=xmatched_lists(cases)

#Choose whether to include plots
plots=False

# #Range of targets
# start=3676
# end=5000 #stops before this index; use this value as start to continue in order

# #Range of tces
# start=100
# end=1324 #stops before this index; use this value as start to continue in order

#Range of FTL
start=4964
end=5000 #stops before this index; use this value as start to continue in order

for i in range(start, end):
    #Chosen TIC ID based on position in the cross-matched SPOC-QLP list
    position_xmatched_ticids=i
    ID=str(int(xmatched_ticids['TICID'][position_xmatched_ticids]))
    TICID='TIC'+ID
    clear_output()
    if ID in excluded_ticids:
        continue
    print(i, TICID, cases)
    
    #Create directory
    directory=f'Output_{cases}/'+f"{TICID}"+'/'
    for option in twomin_options:
        if option!=cases:
            directory_src=f'Output_{option}/'+f"{TICID}"+'/'
            if os.path.isdir(directory_src) and not os.path.isdir(directory):
                shutil.copytree(directory_src, directory, dirs_exist_ok=True)
                file_path=directory+f"{TICID}"+'_output.log'
                with open(file_path, "r") as fr:
                    text = fr.read()
                    new_text=text.replace(option,cases)
                with open(file_path, "w") as fw:
                    fw.write(new_text)
            else:
                Path(directory).mkdir(parents=True, exist_ok=True)
        
                #Open log file
                filename=f"{TICID}"+'_output.log'
                file=open(directory+filename, "w")
                
                print(f'The chosen TIC ID is {TICID}\n', file=file)
                print(f'This is a cross-match between QLP and SPOC {cases}\n',
                      file=file)
    
    qlp_lc_normalized, qlp_lc, qlp_lc_sap, qlp_lc_bkg,\
    spoc_lc_normalized, spoc_lc, spoc_lc_sap, spoc_lc_bkg,\
    median_bkg_diff, flux_up, flux_down, flux_2up,\
    flux_2down = load_QLP_SPOC_lcs()

    calculate_stats()
            
    if plots:
        plot_QLP_SPOC_tests()
                
    file.close()

4999 TIC28191542 FTL
